# BitMask Selective Protection Technique

In [1]:
# General imports
import os
import argparse
import math
from copy import deepcopy
import numpy as np

# Torch imports
import torch
import torch.backends.cudnn as cudnn

# Internal imports
from lib.utils.utils import prYellow
from lib.env import ImportantBitsFloatFirstNEnv, ImportantBitsQuanFirstNEnv
from lib.ddpg import DDPG

#VDCNN imports
from lib.net import VDCNN
from lib.datasets import load_datasets
from torch.utils.data import DataLoader, Dataset
import lmdb

ModuleNotFoundError: No module named 'lib.utils'

In [ ]:
# Models
model_names = ['VDCNN']
print('support models: ', model_names)

In [ ]:
# Suppress warnings (PyTorch UserWarning for non-writeable Tensor)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
def train(num_episode, agent, env, output, args, debug=False):
    '''
    Training function for finding best reward (weight mask), policy, and training cycle via Deep Reinforcement Learning using 
    specified agent and environment.
    
    Parameters:
        num_episode (int) :
            number of cycles to test for
        agent (DDPG) : 
            DDPG object containing actor/critic DRL networks & necessary DRL functions
        env (ImportantBitsEnv) :
            environment object for performing DRL actions with agent
        output (str) : 
            folder for saving model to at regular checkpoints
        args (ArgumentParser) :
            arguments specified at runtime. function-specific args include args.warmup and args.n_update.
        debug=False (bool) :
            specifies whether or not cycle info should be directly printed in addition to being written to an output file
    '''
    # best record
    best_reward = -math.inf
    best_policy = []
    best_episode_num = 0
    acc = 0.0
    ratio = 0.0
    best_history = []

    agent.is_training = True
    step = episode = episode_steps = 0
    episode_reward = 0.
    observation = None
    T = []  # trajectory
    while episode < num_episode:  # counting based on episode
        # reset if it is the start of episode
        if observation is None:
            observation = deepcopy(env.reset())
            agent.reset(observation)
            if episode > args.warmup:
                # decay
                agent.step()

        # agent pick action ...
        # Random action returns value from uniform range [0,1] if warming up
        # Otherwise has agent select action based off of observation
        
        if episode <= args.warmup:
            action = agent.random_action()
        else:
            action = agent.select_action(observation)

        # env response with next_observation, reward, terminate_info
        # Uses action to set policy, and adds to record
        # Note: reward changes based on episode (decay factor), which is why diff reward comes from same policy. Also has diff acc because diff noise added each time. 
        # Believe that it's just generally testing how it holds up to diff noise and gives reward based on robustness
        observation2, reward, done, info = env.step(action)
        observation2 = deepcopy(observation2)
        T.append([reward, deepcopy(observation), deepcopy(observation2), action, done])

        # [optional] save intermediate model
        if episode % int(num_episode / 10) == 0:
            agent.save_model(output)

        # update
        step += 1
        episode_steps += 1
        episode_reward += reward
        observation = deepcopy(observation2)

        debug = True
        
        # info['w_ratio'] is calculated as (curr weight / original weight), with a multiplier of 1/8e6 converting from bits to megabytes
        # Ex:
        #     curr weight = 200 (1/4 of weight as per preserve ratio)
        #     orig weight = 800
        #     weight, as printed: 3.125e-8
        
        if done:  # end of episode
            if debug:
                print('#{}: episode_reward: {:.4f} acc: {:.4f}, weight: {} MB'.format(episode, episode_reward,
                                                                                         info['accuracy'],
                                                                                         info['w_ratio'] * 1. / 8e6))
            text_writer.write(
                '#{}: episode_reward: {:.4f} acc: {:.4f}, weight: {} MB\n'.format(episode, episode_reward,
                                                                                     info['accuracy'],
                                                                                     info['w_ratio'] * 1. / 8e6))
            final_reward = T[-1][0]
            # agent observe and update policy
            for i, (r_t, s_t, s_t1, a_t, done) in enumerate(T):
                agent.observe(final_reward, s_t, a_t, done)
                if episode > args.warmup:
                    for i in range(args.n_update):
                        agent.update_policy()

            agent.memory.append(
                observation,
                agent.select_action(observation),
                0., False
            )

            # Reset
            observation = None
            episode_steps = 0
            episode_reward = 0.
            episode += 1
            T = []

            # Set best reward info
            if final_reward > best_reward:
                best_reward = final_reward
                best_policy = env.strategy
                best_episode_num = episode
                acc = info['accuracy']
                ratio = info['w_ratio']
                best_history.append(['episode: {}'.format(best_episode_num), 'reward: {:.4f}'.format(best_reward), 'acc: {:.4f}'.format(acc), 'ratio: {:.4f}'.format(ratio)])

            text_writer.write('best_reward: {}\n'.format(best_reward))
            text_writer.write('best_policy: {}\n'.format(best_policy))
    text_writer.write('best_accuracy: {}\n'.format(acc))
    text_writer.write('best_ratio: {}\n'.format(ratio))
    text_writer.write('best_episode: {}\n'.format(best_episode_num))
    text_writer.write('best_outcome_history: {}\n'.format(best_history))
    text_writer.close()
    
    return best_policy, best_reward, best_episode_num, acc, ratio, best_history

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description='PyTorch Reinforcement Learning')

    parser.add_argument('--suffix', default=None, type=str, help='suffix to help you remember what experiment you ran')
    # env
    parser.add_argument('--dataset', default='AG_NEWS', type=str, help='dataset to use)')
    parser.add_argument('--dataset_root', default='/home/jovyan/Kunping_Code/datasets/ag_news/vdcnn/test.lmdb', type=str, help='path to dataset)')
    parser.add_argument('--preserve_ratio', default=0.25, type=float, help='preserve ratio of the model size')
    parser.add_argument('--min_bit', default=0, type=int, help='minimum bit to use')
    parser.add_argument('--max_bit', default=32, type=int, help='maximum bit to use')
    parser.add_argument('--n_actions', default=10, type=int, help='size of actions')
    parser.add_argument('--bit', default=32, type=int, help='bitwidth of model to use')
    parser.add_argument('--float_bit', default=32, type=int, help='the bit of full precision float')
    parser.add_argument('--is_pruned', dest='is_pruned', action='store_true')
    # ddpg
    parser.add_argument('--hidden1', default=300, type=int, help='hidden num of first fully connect layer')
    parser.add_argument('--hidden2', default=300, type=int, help='hidden num of second fully connect layer')
    parser.add_argument('--lr_c', default=1e-3, type=float, help='learning rate for actor')
    parser.add_argument('--lr_a', default=1e-4, type=float, help='learning rate for actor')
    parser.add_argument('--warmup', default=30, type=int,
                        help='time without training but only filling the replay memory')
    parser.add_argument('--discount', default=1., type=float, help='')
    parser.add_argument('--bsize', default=128, type=int, help='minibatch size')
    parser.add_argument('--rmsize', default=128, type=int, help='memory size for each layer')
    parser.add_argument('--window_length', default=1, type=int, help='')
    parser.add_argument('--tau', default=0.01, type=float, help='moving average for target network')
    # noise (truncated normal distribution)
    parser.add_argument('--init_delta', default=0.5, type=float,
                        help='initial variance of truncated normal distribution')
    parser.add_argument('--delta_decay', default=0.995, type=float,
                        help='delta decay during exploration')
    parser.add_argument('--n_update', default=1, type=int, help='number of rl to update each time')
    # training
    parser.add_argument('--max_episode_length', default=1e9, type=int, help='')
    parser.add_argument('--output', default='../checkpoints/checkpoint_length/', type=str, help='output dir for drl weights and log file')
    parser.add_argument('--debug', dest='debug', action='store_true')
    parser.add_argument('--init_w', default=0.003, type=float, help='')
    parser.add_argument('--train_episode', default=5000, type=int, help='train iters each timestep')
    parser.add_argument('--epsilon', default=50000, type=int, help='linear decay of exploration policy')
    parser.add_argument('--seed', default=17, type=int, help='')
    # n_worker decreased to alleviate runtime crashes on nautilus cluster
    #parser.add_argument('--n_worker', default=32, type=int, help='number of data loader worker')
    parser.add_argument('--n_worker', default=0, type=int, help='number of data loader worker')
    parser.add_argument('--data_bsize', default=128, type=int, help='number of data batch size')
    parser.add_argument('--finetune_lr', default=0.001, type=float, help='finetune gamma')
    parser.add_argument('--finetune_epoch', default=1, type=int, help='')
    parser.add_argument('--use_top5', default=False, type=bool, help='whether to use top5 acc in reward')
    parser.add_argument('--train_size', default=20000, type=int, help='number of train data size')
    parser.add_argument('--val_size', default=30000, type=int, help='number of val data size')
    parser.add_argument('--resume', default='default', type=str, help='Resuming model path for testing')
    parser.add_argument('--representation', default='float', type=str, help='decide between floating point or fixed point', choices={'fixed', 'float'})
    parser.add_argument('--code', default='ideal', type=str, help='select between ideal code or BCH code', choices={'bch', 'ideal'})
    parser.add_argument('--history', default=True, type=bool, help='whether or not to display output of all best rewards/policy/etc')
    
    # Architecture
    parser.add_argument('--arch', '-a', metavar='ARCH', default='VDCNN', choices=model_names,
                    help='model architecture:' + ' | '.join(model_names) + ' (default: VDCNN)')
    # device options
    parser.add_argument('--gpu_id', default='7', type=str,
                        help='id(s) for CUDA_VISIBLE_DEVICES')

    args = parser.parse_args('')
    base_folder_name = '{}_{}'.format(args.arch, args.dataset)
    if args.suffix is not None:
        base_folder_name = base_folder_name + '_' + args.suffix
    args.output = os.path.join(args.output, base_folder_name)
    if not os.path.exists(args.output):
        os.mkdir(args.output)
    # tfwriter = SummaryWriter(logdir=args.output)
    text_writer = open(os.path.join(args.output, 'log.txt'), 'w')
    print('==> Output path: {}...'.format(args.output))

    # Use CUDA
    # os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_id
    assert torch.cuda.is_available(), 'CUDA is needed for CNN'

    # Set seed so that actions taken will be reproducible 
    if args.seed > 0:
        np.random.seed(args.seed)
        torch.manual_seed(args.seed)
        torch.cuda.manual_seed_all(args.seed)

    # Create model
    # model = models.__dict__[args.arch](pretrained=True)
    if args.dataset == 'AG_NEWS':
        model = VDCNN(n_classes=4, num_embedding=69, embedding_dim=16, depth=9, n_fc_neurons=2048, shortcut=True)
        state_dict = torch.load('results/models/model_epoch_100')
    else:
        raise ValueError('An incorrect dataset value was given')
    model.load_state_dict(state_dict)

    if args.arch.startswith('alexnet') or args.arch.startswith('vgg'):
        model.features = torch.nn.DataParallel(model.features)
        model.cuda()
    else:
        model = torch.nn.DataParallel(model).cuda()

    print('    Total params: %.2fM' % (sum(p.numel() for p in model.parameters())/1000000.0))
    cudnn.benchmark = True
    

    # Set code type
    if args.code == 'bch':
        if args.representation == 'fixed':
            code = (8191, 6787, 110) #fixed point
        if args.representation == 'float':
            code = (8191, 6722, 115) #floating point
    elif args.code == 'ideal':
        code = None

    # Create environment based off of representation type 
    if args.representation == 'fixed':
        env = ImportantBitsQuanFirstNEnv(model, args.dataset, args.dataset_root,
                      compress_ratio=args.preserve_ratio, num_workers=args.n_worker,
                      batch_size=args.data_bsize, args=args, bitwidth=args.bit, is_model_pruned=args.is_pruned, code=code)
    elif args.representation == 'float':
        env = ImportantBitsFloatFirstNEnv(model, args.dataset, args.dataset_root,
                      compress_ratio=args.preserve_ratio, num_workers=args.n_worker,
                      batch_size=args.data_bsize, args=args, bitwidth=args.bit, is_model_pruned=args.is_pruned, code=code)

    # Create agent + requirements and train in environment
    nb_states = env.layer_embedding.shape[1] # number of dimensions (layer type, #in, #out, stride, kernel size, weight size, input feature map size, layer index, max bit)
    nb_actions = 1 # actions for weight and activation quantization
    args.rmsize = args.rmsize * len(env.layer_idx)  # replay memory * num layers to evaluate
    print('** Actual replay buffer size: {}'.format(args.rmsize))
    agent = DDPG(nb_states, nb_actions, len(env.layer_idx), args)

    best_policy, best_reward, best_episode_num, acc, ratio, best_history = train(args.train_episode, agent, env, args.output, args, debug=args.debug)   
    
    # Creates layer mask based off of best_policy for use in noise addition stage
    best_policy_list = []
    for policy in best_policy:
        li = ['1' for index in range(policy)] + ['0' for index in range(args.bit - policy)]
        li = ''.join(li)
        best_policy_list.append(li)
    
    # Output best policy information
    print('Scheme: topbits_{}_{}'.format(args.representation, args.code))
    print('best_episode_num: ', best_episode_num)
    print('best_reward: ', best_reward)
    print('best_policy: ', best_policy)
    print('best_policy (list): ', best_policy_list)
    print('best_accurary: ', acc)
    print('best_ratio: ', ratio)
    
    if args.history:
        print('best_outcome_history: {}'.format(best_history))